# Byzantine-Robust FL-IDS with Blockchain Governance on Kaggle
This notebook runs the Phase E3 diagnostic suite on GPU (AUC, Skew Confound, Honest Flagging, FedAvg Controls, Convergence).

## 1. Verify Environment & Device

In [ ]:
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")
else:
    print("Using High-Performance Multi-Core CPU")

## 2. Unpack Code and Datasets into Working Directory

In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

os.chdir("/kaggle/working")

# 1. Extract any zip files found in /kaggle/input (e.g. data.zip, src.zip)
for z in Path("/kaggle/input").rglob("*.zip"):
    print(f"Extracting {z} into /kaggle/working...")
    with zipfile.ZipFile(z, "r") as zf:
        zf.extractall("/kaggle/working")

# 2. Copy directories from capstone-code if present as directories
code_in = Path("/kaggle/input/capstone-code")
if code_in.exists():
    for item in ["src", "configs", "tests"]:
        src_item = code_in / item
        dst_item = Path("/kaggle/working") / item
        if src_item.is_dir() and not dst_item.exists():
            shutil.copytree(src_item, dst_item)
            print(f"Copied {item} to /kaggle/working/{item}")

# 3. Symlink dataset dev partitions and processed splits
p_dev = None
for p in Path("/kaggle").rglob("client_00.parquet"):
    if "partitions" in str(p):
        p_dev = p.parent
        break

pr_dev = None
for p in Path("/kaggle").rglob("test.parquet"):
    if "processed" in str(p):
        pr_dev = p.parent
        break

if p_dev and not Path("/kaggle/working/data/partitions/dev").exists():
    os.makedirs("/kaggle/working/data/partitions", exist_ok=True)
    os.system(f"ln -sfn '{p_dev}' /kaggle/working/data/partitions/dev")
    print(f"Linked partitions/dev -> {p_dev}")

if pr_dev and not Path("/kaggle/working/data/processed/dev").exists():
    os.makedirs("/kaggle/working/data/processed", exist_ok=True)
    os.system(f"ln -sfn '{pr_dev}' /kaggle/working/data/processed/dev")
    print(f"Linked processed/dev -> {pr_dev}")

# Copy metadata if present in input
meta_found = False
for m in Path("/kaggle").rglob("feature_stats_dev.json"):
    os.makedirs("/kaggle/working/data/metadata", exist_ok=True)
    shutil.copy2(m, "/kaggle/working/data/metadata/feature_stats_dev.json")
    meta_found = True
    print(f"Linked metadata from {m}")
    break

print("\n=== Verification ===")
!ls -la /kaggle/working
!ls /kaggle/working/data/partitions/dev | head -n 10
!ls /kaggle/working/data/processed/dev

## 3. Run Test Suite

In [ ]:
!PYTHONPATH=. pytest tests/test_determinism.py tests/test_harness.py tests/test_metrics.py tests/test_variants.py

## 4. Run High-Performance Phase E3 Diagnostic Suite

In [ ]:
!PYTHONPATH=. python src/experiments/run_phase_e3_diagnosis.py --workers 4 --rounds 30

## 5. Package Results

In [ ]:
!zip -r /kaggle/working/phase_e3_results.zip results/runs/phase_e3_diagnosis/
print("Results ready at /kaggle/working/phase_e3_results.zip")
!ls -lh /kaggle/working/phase_e3_results.zip